# CASMI frozen local-validation winner
Offline inference. Attach the generated bundle, competition and COCONUT. The bundle supplies the RDKit wheel.
Local validation results are in REPORT.md. Latest competition rules require manual verification.

In [ ]:
from pathlib import Path
import sys, subprocess, json, importlib.metadata
# Attach the generated artifact directory as a private Kaggle Dataset and attach
# the competition files and the same COCONUT dataset used during validation.
import zipfile
roots = list(Path('/kaggle/input').rglob('final_selection.json'))
if len(roots) == 1:
    bundle = roots[0].parent
else:
    archives = list(Path('/kaggle/input').rglob('casmi_cpu_bundle.zip'))
    assert len(archives) == 1, f'Expected one model archive, found {archives}'
    bundle = Path('/kaggle/working/casmi_runtime')
    bundle.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(bundle)
import hashlib
for name, expected in json.loads((bundle / 'SHA256SUMS.json').read_text()).items():
    assert hashlib.sha256((bundle / name).read_bytes()).hexdigest() == expected, name
required = json.loads((bundle / 'runtime_versions.json').read_text())
try:
    installed_rdkit = importlib.metadata.version('rdkit')
except importlib.metadata.PackageNotFoundError:
    installed_rdkit = None
if installed_rdkit != required['rdkit']:
    wheels = list((bundle / 'wheels').glob('rdkit*.whl'))
    assert len(wheels) == 1, 'Bundle must include the training-version RDKit wheel'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', str(wheels[0])])
import torch
sys.path.insert(0, str(bundle))
from casmi_ml.inference import predict
predict(bundle / 'final_selection.json', '/kaggle/input/competition',
        '/kaggle/input/coconut_structures.parquet', '/kaggle/working/submission.csv')
